# AI Hub Emotion TTS Fine-tuning - Fish Speech 1.5

This notebook is the reset path after the OpenAudio S1-mini compatibility issues. It uses the stable Fish Speech 1.5 family end to end:

- Code: `fishaudio/fish-speech` tag `v1.5.1`
- Checkpoint: `fishaudio/fish-speech-1.5`
- VQ: `firefly_gan_vq`
- Fine-tune: `text2semantic_finetune` + LoRA `r_8_alpha_16`

Run this notebook from top to bottom in a fresh Colab runtime.

## 0. Runtime

Use `Runtime > Change runtime type > GPU`. Start with `RUN_MODE = "smoke"`. After smoke training succeeds, change to `RUN_MODE = "full"` and rerun from dataset preparation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_MODE = 'smoke'  # 'smoke' or 'full'
assert RUN_MODE in {'smoke', 'full'}

DRIVE_ROOT = Path('/content/drive/MyDrive/gyul-ai/emotion-tts')
ARCHIVE_DIR = DRIVE_ROOT / 'archive'
RAW_DIR = DRIVE_ROOT / 'raw'
PROCESSED_ROOT = DRIVE_ROOT / 'processed'
PROTO_ROOT = DRIVE_ROOT / 'protos'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
RESULTS_ROOT = DRIVE_ROOT / 'results'
SAMPLES_ROOT = DRIVE_ROOT / 'generated_samples'

REPO_DIR = Path('/content/Gyul-AI-Repository')
FISH_DIR = Path('/content/fish-speech-v15')
BASE_CKPT = CHECKPOINT_ROOT / 'fish-speech-1.5'

RUN_PROCESSED_DIR = PROCESSED_ROOT / f'fish15_{RUN_MODE}'
RUN_PROTO_DIR = PROTO_ROOT / f'fish15_{RUN_MODE}'
PROJECT_NAME = f'aihub_emotion_fish15_lora_{RUN_MODE}'
RUN_RESULTS_DIR = RESULTS_ROOT / PROJECT_NAME
MERGED_CKPT = CHECKPOINT_ROOT / f'fish-speech-1.5-aihub-emotion-{RUN_MODE}'

if RUN_MODE == 'smoke':
    SAMPLES_PER_EMOTION = 10
    TRAIN_MAX_STEPS = 20
    CHECKPOINT_EVERY = 20
else:
    SAMPLES_PER_EMOTION = 300
    TRAIN_MAX_STEPS = 1500
    CHECKPOINT_EVERY = 100

for path in [ARCHIVE_DIR, RAW_DIR, PROCESSED_ROOT, PROTO_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, SAMPLES_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('RUN_MODE:', RUN_MODE)
print('samples_per_emotion:', SAMPLES_PER_EMOTION)
print('train_max_steps:', TRAIN_MAX_STEPS)
print('processed:', RUN_PROCESSED_DIR)
print('protos:', RUN_PROTO_DIR)
print('checkpoint:', BASE_CKPT)

## 1. AI Hub Dataset

Upload the AI Hub archive to:

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/archive/emotion_voice_dataset.zip
```

If the dataset is already extracted under `raw`, this cell will reuse it.

In [ ]:
import subprocess

ARCHIVE_PATH = ARCHIVE_DIR / 'emotion_voice_dataset.zip'
if ARCHIVE_PATH.exists():
    print('Extracting:', ARCHIVE_PATH)
    subprocess.run(['unzip', '-oq', str(ARCHIVE_PATH), '-d', str(RAW_DIR)], check=True)
else:
    print('Archive not found. Reusing extracted files under:', RAW_DIR)

expected_prefixes = ('ang_', 'dis_', 'fea_', 'hap_', 'neu_', 'sad_', 'sur_')
candidates = []
for directory in [RAW_DIR, *RAW_DIR.rglob('*')]:
    if not directory.is_dir():
        continue
    names = [child.name for child in directory.iterdir() if child.is_dir()]
    matched = sum(any(name.startswith(prefix) for name in names) for prefix in expected_prefixes)
    if matched >= 4:
        candidates.append((matched, directory))

assert candidates, f'AI Hub emotion folders were not found under {RAW_DIR}'
AIHUB_INPUT_DIR = sorted(candidates, key=lambda item: (-item[0], len(str(item[1]))))[0][1]
print('AIHUB_INPUT_DIR:', AIHUB_INPUT_DIR)
for child in sorted(AIHUB_INPUT_DIR.iterdir()):
    if child.is_dir():
        print('-', child.name)

## 2. Install Fish Speech 1.5 Stack

This installs Fish Speech `v1.5.1` without pulling the incompatible latest Fish Speech code.

In [ ]:
REPO_URL = 'https://github.com/novvvv/Gyul-AI-Repository.git'
BRANCH = 'MaTuna/tts'

%cd /content
!rm -rf "{REPO_DIR}"
!git clone --branch "{BRANCH}" "{REPO_URL}" "{REPO_DIR}"
%cd /content/Gyul-AI-Repository
!git status --short --branch
!test -f scripts/prepare_aihub_emotion_dataset.py
!test -f configs/emotion_tags.yaml

In [ ]:
FISH_REPO = 'https://github.com/fishaudio/fish-speech.git'
FISH_TAG = 'v1.5.1'

%cd /content
!apt-get update -y
!apt-get install -y portaudio19-dev libasound2-dev ffmpeg libsox-dev libsndfile1
!rm -rf "{FISH_DIR}"
!git clone --branch "{FISH_TAG}" --depth 1 "{FISH_REPO}" "{FISH_DIR}"
%cd /content/fish-speech-v15

import os, sys
os.environ['PYTHONPATH'] = f"{FISH_DIR}:{os.environ.get('PYTHONPATH', '')}"
if str(FISH_DIR) not in sys.path:
    sys.path.insert(0, str(FISH_DIR))

!python -m pip install --upgrade pip
!python -m pip install -e . --no-deps
!python -m pip install -U \
  'numpy<=1.26.4' 'transformers>=4.45.2,<4.58' 'datasets==2.18.0' \
  'lightning>=2.1.0' 'hydra-core>=1.3.2' 'tensorboard>=2.14.1' \
  'natsort>=8.4.0' 'einops>=0.7.0' 'librosa>=0.10.1' \
  'rich>=13.5.3,<14' 'wandb>=0.15.11' 'grpcio>=1.58.0' \
  'loguru>=0.6.0' 'loralib>=0.1.2' 'pyrootutils>=1.0.4' \
  'vector_quantize_pytorch==1.14.24' 'resampy>=0.4.3' \
  'einx[torch]==0.2.2' 'zstandard>=0.22.0' pydub \
  'opencc-python-reimplemented==0.1.7' ormsgpack 'tiktoken>=0.8.0' \
  'pydantic==2.9.2' cachetools soundfile
!python -m pip install -U 'protobuf>=4.25.1,<5'
!python -m pip uninstall -y torchvision

import torch, torchaudio, fish_speech
print('fish_speech:', fish_speech.__file__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('torchaudio:', torchaudio.__version__)
!git log -1 --oneline

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime is required.'
print('gpu:', torch.cuda.get_device_name(0))
print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Download Fish Speech 1.5 Checkpoint

In [ ]:
!hf download fishaudio/fish-speech-1.5 --local-dir "{BASE_CKPT}" --max-workers 1
!find "{BASE_CKPT}" -maxdepth 1 -type f -printf '%f\n' | sort

required = [
    'model.pth',
    'tokenizer.tiktoken',
    'special_tokens.json',
    'firefly-gan-vq-fsq-8x1024-21hz-generator.pth',
]
missing = [name for name in required if not (BASE_CKPT / name).exists()]
assert not missing, missing

In [ ]:
%cd /content/fish-speech-v15
from fish_speech.tokenizer import FishTokenizer

tokenizer = FishTokenizer.from_pretrained(BASE_CKPT)
print('semantic_count:', len(tokenizer.semantic_id_to_token_id))
print('semantic_range:', tokenizer.semantic_begin_id, tokenizer.semantic_end_id)
assert len(tokenizer.semantic_id_to_token_id) == 1024

## 4. Prepare Dataset

This creates fresh Fish Speech 1.5 data folders. Do not reuse S1-mini `.npy` or protobuf outputs.

In [ ]:
%cd /content/Gyul-AI-Repository
!python scripts/prepare_aihub_emotion_dataset.py \
  --input-dir "{AIHUB_INPUT_DIR}" \
  --output-dir "{RUN_PROCESSED_DIR}" \
  --samples-per-emotion {SAMPLES_PER_EMOTION} \
  --overwrite

for emotion_dir in sorted(RUN_PROCESSED_DIR.iterdir()):
    if emotion_dir.is_dir():
        print(emotion_dir.name, 'wav:', len(list(emotion_dir.glob('*.wav'))), 'lab:', len(list(emotion_dir.glob('*.lab'))))

## 5. Extract VQ Tokens

In [ ]:
%cd /content/fish-speech-v15

from pathlib import Path

# Compatibility for newer torchaudio versions where list_audio_backends() was removed.
vq_script = Path('/content/fish-speech-v15/tools/vqgan/extract_vq.py')
vq_source = vq_script.read_text(encoding='utf-8')
shim = '''import torchaudio\n\nif not hasattr(torchaudio, "list_audio_backends"):\n    def _fish_list_audio_backends():\n        return ["soundfile"]\n    torchaudio.list_audio_backends = _fish_list_audio_backends\n'''
if '_fish_list_audio_backends' not in vq_source:
    assert 'import torchaudio\n' in vq_source
    vq_script.write_text(vq_source.replace('import torchaudio\n', shim, 1), encoding='utf-8')
    print('patched torchaudio backend shim')
else:
    print('torchaudio backend shim already present')

VQ_BATCH_SIZE = 16
!python tools/vqgan/extract_vq.py "{RUN_PROCESSED_DIR}" \
  --num-workers 1 \
  --batch-size {VQ_BATCH_SIZE} \
  --config-name firefly_gan_vq \
  --checkpoint-path "{BASE_CKPT / 'firefly-gan-vq-fsq-8x1024-21hz-generator.pth'}"

## 6. Build Protobuf Dataset

In [ ]:
%cd /content/fish-speech-v15
!python -m pip install -U 'protobuf>=4.25.1,<5'
!rm -rf "{RUN_PROTO_DIR}"
!python tools/llama/build_dataset.py \
  --input "{RUN_PROCESSED_DIR}" \
  --output "{RUN_PROTO_DIR}" \
  --text-extension .lab \
  --num-workers 4

!rm -rf data/protos
!mkdir -p data
!ln -s "{RUN_PROTO_DIR}" data/protos
!find "{RUN_PROTO_DIR}" -maxdepth 1 -type f -name '*.protos' -print
!ls -la data/protos

## 7. LoRA Train

In [ ]:
%cd /content/fish-speech-v15

BATCH_SIZE = 2
GRAD_ACCUM = 8
gpu_name = torch.cuda.get_device_name(0).lower()
PRECISION = '16-mixed' if 't4' in gpu_name else 'bf16-true'

print('project:', PROJECT_NAME)
print('precision:', PRECISION)
print('max_steps:', TRAIN_MAX_STEPS)

!python fish_speech/train.py --config-name text2semantic_finetune \
  project={PROJECT_NAME} \
  pretrained_ckpt_path="{BASE_CKPT}" \
  data.batch_size={BATCH_SIZE} \
  trainer.accumulate_grad_batches={GRAD_ACCUM} \
  trainer.max_steps={TRAIN_MAX_STEPS} \
  +trainer.num_sanity_val_steps=0 \
  trainer.limit_val_batches=0 \
  trainer.val_check_interval={CHECKPOINT_EVERY} \
  callbacks.model_checkpoint.every_n_train_steps={CHECKPOINT_EVERY} \
  ++callbacks.model_checkpoint.dirpath="{RUN_RESULTS_DIR / 'checkpoints'}" \
  trainer.precision={PRECISION} \
  hydra.run.dir="{RUN_RESULTS_DIR}" \
  +lora@model.model.lora_config=r_8_alpha_16

## 8. Merge LoRA

In [ ]:
%cd /content/fish-speech-v15

ckpt_dir = RUN_RESULTS_DIR / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('*.ckpt'), key=lambda path: path.stat().st_mtime)
print('checkpoint_dir:', ckpt_dir)
for ckpt in ckpts:
    print('-', ckpt)
assert ckpts, f'No .ckpt files found in {ckpt_dir}'

LORA_CKPT = ckpts[-1]
print('selected_lora_ckpt:', LORA_CKPT)
print('merged_output:', MERGED_CKPT)

!rm -rf "{MERGED_CKPT}"
!python tools/llama/merge_lora.py \
  --lora-config r_8_alpha_16 \
  --base-weight "{BASE_CKPT}" \
  --lora-weight "{LORA_CKPT}" \
  --output "{MERGED_CKPT}"

!find "{MERGED_CKPT}" -maxdepth 1 -type f -printf '%f\n' | sort

## 9. Generate Emotion Sample

After LoRA merge, change only `EMOTION` and `TEXT` to generate a short wav sample from the fine-tuned model.

In [ ]:
%cd /content/fish-speech-v15

from pathlib import Path
import subprocess
from IPython.display import Audio, display

EMOTION_TAGS = {
    'neutral': '(indifferent)',
    'happy': '(happy)',
    'sad': '(sad)',
    'angry': '(angry)',
    'anxious': '(anxious)',
    'hurt': '(painful)',
    'embarrassed': '(embarrassed)',
}

# Change these two values for each test.
EMOTION = 'happy'
TEXT = '오늘 정말 잘했어. 조금만 더 힘내보자.'

assert EMOTION in EMOTION_TAGS, f'Unknown emotion: {EMOTION}. Use one of {sorted(EMOTION_TAGS)}'

DRIVE_ROOT = Path('/content/drive/MyDrive/gyul-ai/emotion-tts')
BASE_CKPT = DRIVE_ROOT / 'checkpoints' / 'fish-speech-1.5'
MERGED_CKPT = DRIVE_ROOT / 'checkpoints' / 'fish-speech-1.5-aihub-emotion-full'
DECODER_CKPT = BASE_CKPT / 'firefly-gan-vq-fsq-8x1024-21hz-generator.pth'
OUT_DIR = DRIVE_ROOT / 'generated_samples' / 'fish15_full_test'
SEMANTIC_DIR = OUT_DIR / 'semantic'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

assert (MERGED_CKPT / 'model.pth').exists(), f'Merged model not found: {MERGED_CKPT}'
assert DECODER_CKPT.exists(), f'Decoder checkpoint not found: {DECODER_CKPT}'

for old_code in SEMANTIC_DIR.glob('codes_*.npy'):
    old_code.unlink()

tagged_text = f"{EMOTION_TAGS[EMOTION]} {TEXT.strip()}"
out_wav = OUT_DIR / f'{EMOTION}_sample.wav'
print('tagged_text:', tagged_text)
print('output:', out_wav)

def run_checked(command):
    print('\n$', ' '.join(map(str, command)))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    result.check_returncode()

run_checked([
    'python', 'fish_speech/models/text2semantic/inference.py',
    '--text', tagged_text,
    '--checkpoint-path', str(MERGED_CKPT),
    '--half',
    '--num-samples', '1',
    '--output-dir', str(SEMANTIC_DIR),
])

codes = sorted(SEMANTIC_DIR.glob('codes_*.npy'))
assert codes, f'No semantic token generated in {SEMANTIC_DIR}'
codes_path = codes[0]
print('semantic token:', codes_path)

run_checked([
    'python', 'fish_speech/models/vqgan/inference.py',
    '-i', str(codes_path),
    '-o', str(out_wav),
    '--config-name', 'firefly_gan_vq',
    '--checkpoint-path', str(DECODER_CKPT),
])

assert out_wav.exists(), f'Generated wav not found: {out_wav}'
print('saved:', out_wav)
display(Audio(str(out_wav)))